# 🌦️ Análisis de Regresión Lineal — Clima Colombia
**Pipeline:** PostgreSQL → EDA → Regresión Simple → Regresión Múltiple → VIF → Diagnósticos → Métricas

**Variable objetivo:** `temperatura`  
**Predictores:** `humedad`, `velocidad_viento`, `sensacion_termica`

## 0. Imports y configuración

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from scripts.database import engine

os.makedirs('../data/graficas', exist_ok=True)
GRAFICAS = '../data/graficas'

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Imports OK')

## 1. Extracción desde PostgreSQL

In [ ]:
query = """
    SELECT
        c.nombre        AS ciudad,
        c.pais,
        c.latitud,
        c.longitud,
        r.temperatura,
        r.sensacion_termica,
        r.humedad,
        r.velocidad_viento,
        r.descripcion,
        r.fecha_extraccion
    FROM registros_clima r
    JOIN ciudades c ON c.id = r.ciudad_id
    ORDER BY r.fecha_extraccion
"""

df = pd.read_sql(query, engine)
df['fecha_extraccion'] = pd.to_datetime(df['fecha_extraccion'])
print(f'Registros cargados: {len(df)}')
df.head()

## 2. EDA — Exploración de datos

In [ ]:
print('=== Info general ===')
print(df.info())
print('\n=== Estadísticas descriptivas ===')
df[['temperatura','sensacion_termica','humedad','velocidad_viento']].describe().round(2)

In [ ]:
# Histogramas
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
cols = ['temperatura', 'sensacion_termica', 'humedad', 'velocidad_viento']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for ax, col, color in zip(axes.flat, cols, colors):
    ax.hist(df[col].dropna(), bins=25, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(f'Distribución: {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')

plt.suptitle('Histogramas — Variables Climáticas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAFICAS}/01_histogramas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots por ciudad
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, col in zip(axes.flat, cols):
    df.boxplot(column=col, by='ciudad', ax=ax, grid=False)
    ax.set_title(col)
    ax.set_xlabel('Ciudad')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Boxplots por Ciudad', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAFICAS}/02_boxplots_ciudad.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mapa de correlación
num_cols = ['temperatura', 'sensacion_termica', 'humedad', 'velocidad_viento', 'latitud', 'longitud']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('Mapa de Correlación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAFICAS}/03_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots vs temperatura
predictores = ['humedad', 'velocidad_viento', 'sensacion_termica']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, pred in zip(axes, predictores):
    ax.scatter(df[pred], df['temperatura'], alpha=0.4, s=20)
    m, b = np.polyfit(df[pred].dropna(), df['temperatura'].dropna(), 1)
    x_line = np.linspace(df[pred].min(), df[pred].max(), 100)
    ax.plot(x_line, m * x_line + b, 'r--', linewidth=1.5)
    ax.set_xlabel(pred)
    ax.set_ylabel('temperatura')
    ax.set_title(f'temperatura vs {pred}')

plt.suptitle('Scatter Plots — Temperatura vs Predictores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAFICAS}/04_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preparación de datos

In [ ]:
FEATURES = ['humedad', 'velocidad_viento', 'sensacion_termica']
TARGET   = 'temperatura'

df_model = df[FEATURES + [TARGET]].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 4. Regresión Lineal Simple (mejor predictor)

In [ ]:
# Mejor predictor por correlación
best_pred = df_model[FEATURES].corrwith(y).abs().idxmax()
print(f'Mejor predictor simple: {best_pred}')

X_simple = sm.add_constant(df_model[[best_pred]])
modelo_simple = sm.OLS(y, X_simple).fit()
print(modelo_simple.summary())

In [ ]:
# Gráfica regresión simple
y_pred_simple = modelo_simple.predict(X_simple)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Ajuste
axes[0].scatter(df_model[best_pred], y, alpha=0.4, s=20, label='Datos')
axes[0].plot(df_model[best_pred], y_pred_simple, 'r-', linewidth=2, label='Regresión')
axes[0].set_xlabel(best_pred)
axes[0].set_ylabel('temperatura')
axes[0].set_title(f'Regresión Simple: temperatura ~ {best_pred}')
axes[0].legend()

# Residuos
residuos_simple = y - y_pred_simple
axes[1].scatter(y_pred_simple, residuos_simple, alpha=0.4, s=20)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Valores ajustados')
axes[1].set_ylabel('Residuos')
axes[1].set_title('Residuos vs Ajustados (Simple)')

plt.tight_layout()
plt.savefig(f'{GRAFICAS}/05_regresion_simple.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Regresión Lineal Múltiple

In [ ]:
X_mult = sm.add_constant(df_model[FEATURES])
modelo_mult = sm.OLS(y, X_mult).fit()
print(modelo_mult.summary())

In [ ]:
# Visualizar coeficientes
coefs = modelo_mult.params.drop('const')
conf  = modelo_mult.conf_int().drop('const')
err   = conf[1] - coefs

fig, ax = plt.subplots(figsize=(8, 4))
colors_coef = ['#E53935' if v < 0 else '#43A047' for v in coefs]
bars = ax.barh(coefs.index, coefs.values, xerr=err.values,
               color=colors_coef, alpha=0.8, capsize=5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coeficiente (con IC 95%)')
ax.set_title('Coeficientes — Regresión Múltiple', fontweight='bold')

for bar, val in zip(bars, coefs.values):
    ax.text(val + (0.02 if val >= 0 else -0.02), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig(f'{GRAFICAS}/06_coeficientes.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cálculo de VIF (Multicolinealidad)

In [ ]:
X_vif = df_model[FEATURES].copy()

vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif_data['Interpretación'] = vif_data['VIF'].apply(
    lambda v: '✅ Sin multicolinealidad' if v < 5
    else ('⚠️ Moderada' if v < 10 else '❌ Alta multicolinealidad')
)
print(vif_data.to_string(index=False))

# Gráfica VIF
fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ['#43A047' if v < 5 else ('#FF9800' if v < 10 else '#E53935') for v in vif_data['VIF']]
ax.barh(vif_data['Variable'], vif_data['VIF'], color=bar_colors, alpha=0.85)
ax.axvline(5,  color='orange', linestyle='--', label='Umbral moderado (5)')
ax.axvline(10, color='red',    linestyle='--', label='Umbral alto (10)')
ax.set_xlabel('VIF')
ax.set_title('Factor de Inflación de Varianza (VIF)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{GRAFICAS}/07_vif.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Diagnósticos visuales y tests estadísticos

In [ ]:
y_pred_mult = modelo_mult.predict(X_mult)
residuos    = modelo_mult.resid

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Residuos vs Ajustados
axes[0,0].scatter(y_pred_mult, residuos, alpha=0.4, s=20)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_xlabel('Valores ajustados')
axes[0,0].set_ylabel('Residuos')
axes[0,0].set_title('Residuos vs Ajustados')

# 2. Q-Q Plot (normalidad)
sm.qqplot(residuos, line='s', ax=axes[0,1], alpha=0.5)
axes[0,1].set_title('Q-Q Plot (Normalidad de Residuos)')

# 3. Histograma de residuos
axes[1,0].hist(residuos, bins=30, color='#2196F3', alpha=0.75, edgecolor='white', density=True)
xmin, xmax = axes[1,0].get_xlim()
x_norm = np.linspace(xmin, xmax, 100)
axes[1,0].plot(x_norm, stats.norm.pdf(x_norm, residuos.mean(), residuos.std()),
               'r-', linewidth=2, label='Normal teórica')
axes[1,0].set_title('Distribución de Residuos')
axes[1,0].legend()

# 4. Scale-Location (homocedasticidad)
sqrt_abs_res = np.sqrt(np.abs(residuos))
axes[1,1].scatter(y_pred_mult, sqrt_abs_res, alpha=0.4, s=20)
axes[1,1].set_xlabel('Valores ajustados')
axes[1,1].set_ylabel('√|Residuos|')
axes[1,1].set_title('Scale-Location (Homocedasticidad)')

plt.suptitle('Diagnósticos del Modelo Múltiple', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAFICAS}/08_diagnosticos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tests estadísticos
print('═'*55)
print('  TESTS ESTADÍSTICOS')
print('═'*55)

# Normalidad — Shapiro-Wilk
sample = residuos.sample(min(500, len(residuos)), random_state=42)
stat_sw, p_sw = stats.shapiro(sample)
print(f'\nShapiro-Wilk (normalidad residuos):')
print(f'  W={stat_sw:.4f}  p={p_sw:.4f}  → {"✅ Normal" if p_sw > 0.05 else "⚠️ No normal"}')

# Homocedasticidad — Breusch-Pagan
bp_stat, bp_p, _, _ = het_breuschpagan(residuos, X_mult)
print(f'\nBreusch-Pagan (homocedasticidad):')
print(f'  LM={bp_stat:.4f}  p={bp_p:.4f}  → {"✅ Homocedasticidad" if bp_p > 0.05 else "⚠️ Heterocedasticidad"}')

# Durbin-Watson (autocorrelación)
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(residuos)
print(f'\nDurbin-Watson (autocorrelación):')
print(f'  DW={dw:.4f}  → {"✅ Sin autocorrelación" if 1.5 < dw < 2.5 else "⚠️ Posible autocorrelación"}')
print('═'*55)

## 8. Tabla comparativa de métricas

In [ ]:
# Modelo simple (sklearn)
reg_simple = LinearRegression()
reg_simple.fit(X_train[[best_pred]], y_train)
y_pred_s_test = reg_simple.predict(X_test[[best_pred]])

cv_simple = cross_val_score(reg_simple, X[[best_pred]], y, cv=5, scoring='r2')

# Modelo múltiple (sklearn)
reg_mult = LinearRegression()
reg_mult.fit(X_train_sc, y_train)
y_pred_m_test = reg_mult.predict(X_test_sc)

cv_mult = cross_val_score(reg_mult, scaler.fit_transform(X), y, cv=5, scoring='r2')

def metricas(y_true, y_pred, cv_scores, nombre):
    return {
        'Modelo':   nombre,
        'R²':       round(r2_score(y_true, y_pred), 4),
        'R² CV':    round(cv_scores.mean(), 4),
        'RMSE':     round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE':      round(mean_absolute_error(y_true, y_pred), 4),
        'R² adj':   round(modelo_mult.rsquared_adj if nombre != 'Simple' else modelo_simple.rsquared_adj, 4),
        'AIC':      round(modelo_mult.aic if nombre != 'Simple' else modelo_simple.aic, 2),
        'BIC':      round(modelo_mult.bic if nombre != 'Simple' else modelo_simple.bic, 2),
    }

tabla = pd.DataFrame([
    metricas(y_test, y_pred_s_test, cv_simple, 'Simple'),
    metricas(y_test, y_pred_m_test, cv_mult,   'Múltiple'),
])

print('\n' + '═'*65)
print('  TABLA COMPARATIVA DE MÉTRICAS')
print('═'*65)
print(tabla.to_string(index=False))
print('═'*65)
tabla

In [ ]:
# Gráfica comparativa de métricas
fig, axes = plt.subplots(1, 3, figsize=(13, 5))

metricas_plot = ['R²', 'RMSE', 'MAE']
for ax, met in zip(axes, metricas_plot):
    vals = tabla.set_index('Modelo')[met]
    bars = ax.bar(vals.index, vals.values, color=['#2196F3', '#4CAF50'], alpha=0.85, width=0.4)
    ax.set_title(met, fontweight='bold')
    ax.set_ylim(0, vals.max() * 1.25)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + vals.max()*0.02,
                f'{val:.4f}', ha='center', fontsize=10)

plt.suptitle('Comparativa de Modelos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAFICAS}/09_comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Todas las gráficas guardadas en {GRAFICAS}/')